# `cp_shared` — Full Backend Walkthrough

This notebook is the **single source of truth** for all backend logic. The dashboard (`dashboard.py`) imports code **directly from this notebook** at runtime — there is no separate `.py` file needed.

### How it works
- `dashboard.py` reads this notebook's JSON, extracts all code cells, and executes them as a Python module
- You edit code here, restart the dashboard, and your changes take effect immediately
- You can also run cells interactively for testing and exploration

---

## 1. Imports & Setup

Standard library and third-party imports. We use:
- **NumPy / Pandas** for data manipulation
- **Requests** to check if remote CSV URLs exist
- **scikit-learn** for Linear Regression and Random Forest models
- **warnings** suppressed to keep output clean

In [ ]:
import warnings
import math
from typing import Any, Dict, List, Union
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

## 2. Constants & Configuration

All the global settings that control the pipeline:

| Constant | Purpose |
|---|---|
| `BASE_DATA_URLS` | Known download links for each year's SINAN dengue CSV (2021–2026) |
| `DATA_SOURCE_LABEL` | Official attribution string for the data source |
| `CACHE_DIR` / `WEEKLY_CACHE_PATH` | Local `.data_cache/` folder where processed data is stored as parquet — avoids re-downloading ~500 MB of CSVs on every run |
| `TARGET_UF_CODE` | Brazilian state code **32** = Espírito Santo |
| `ANALYSIS_START` | Only use data from Jan 2023 onward for the dashboard analysis window |
| `FEATURE_COLS` | The 4 features used by the ML models: `lag1`, `lag4`, `week_sin`, `week_cos` |
| `MIN_BACKTEST_TRAIN` | Minimum 26 weeks of training data before backtesting starts |
| `TREND_THRESHOLD` | 10% change triggers a "Rising" or "Falling" label |
| `BAND_QUANTILES` | 25th–75th percentile of residuals used for safety bands |
| `DEFAULT_HOLDOUT_WEEKS` | 52-week holdout for model evaluation |
| `DEFAULT_SELECTION_WEEKS` | Most recent 8 weeks used for model selection scoring |

In [ ]:
import os as _os

BASE_DATA_URLS = {
    2021: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR21.csv.zip",
    2022: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR22.csv.zip",
    2023: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR23.csv.zip",
    2024: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR24.csv.zip",
    2025: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR25.csv.zip",
    2026: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR26.csv.zip",
}
DATA_URL_TEMPLATE = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR{year_suffix}.csv.zip"

# ── Data source metadata ──
DATA_SOURCE_LABEL = "SINAN/Dengue — Brazilian Ministry of Health"
DATA_SOURCE_API_URL = "https://apidadosabertos.saude.gov.br/arboviroses/dengue"
DATA_SOURCE_PORTAL_URL = "https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue/resource/a9b73910-f233-417b-85c9-95230c269e1c"

# ── Local cache paths ──
_MODULE_DIR = _os.path.dirname(_os.path.abspath(__file__)) if "__file__" in dir() else _os.getcwd()
CACHE_DIR = _os.path.join(_MODULE_DIR, ".data_cache")
# Legacy paths (kept for backwards-compat migration of default UF)
WEEKLY_CACHE_PATH = _os.path.join(CACHE_DIR, "weekly_notifications.parquet")
RAW_CACHE_PATH = _os.path.join(CACHE_DIR, "raw_notifications.parquet")
CACHE_META_PATH = _os.path.join(CACHE_DIR, "cache_meta.json")

# ── Selectable areas (IBGE UF code → state name) ──
UF_OPTIONS: Dict[int, str] = {
    32: "Espírito Santo",
    33: "Rio de Janeiro",
    31: "Minas Gerais",
    35: "São Paulo",
    29: "Bahia",
}

TARGET_UF_CODE = 32
TARGET_LABEL = "Espírito Santo"
ANALYSIS_START = pd.Timestamp("2023-01-02")
FEATURE_COLS = ["lag1", "lag4", "week_sin", "week_cos"]
MIN_BACKTEST_TRAIN = 26
TREND_THRESHOLD = 0.10
BAND_QUANTILES = (0.25, 0.75)
DEFAULT_HOLDOUT_WEEKS = 52
DEFAULT_SELECTION_WEEKS = 8
_AVAILABLE_DATA_URLS_CACHE: Union[None, Dict[int, str]] = None


def _uf_cache_paths(uf_code: int = TARGET_UF_CODE) -> Dict[str, str]:
    """Return per-UF cache file paths."""
    return {
        "weekly": _os.path.join(CACHE_DIR, f"weekly_{uf_code}.parquet"),
        "raw": _os.path.join(CACHE_DIR, f"raw_{uf_code}.parquet"),
        "meta": _os.path.join(CACHE_DIR, f"meta_{uf_code}.json"),
    }

## 3. Data URL Discovery

These helper functions handle finding which yearly data files are available on the Brazilian Ministry of Health's S3 bucket:

- **`_build_year_url()`** — constructs the download URL for a given year (e.g., 2026 → `DENGBR26.csv.zip`)
- **`_url_exists()`** — does a HEAD request (falling back to GET) to check if a URL is live
- **`get_available_data_urls()`** — starts with the known 2021–2025 URLs, then probes for any newer years. Results are cached so we only check once per session.

In [ ]:
def _build_year_url(year: int) -> str:
    return DATA_URL_TEMPLATE.format(year_suffix=str(year)[-2:])


def _url_exists(url: str, timeout: int = 8) -> bool:
    try:
        response = requests.head(url, allow_redirects=True, timeout=timeout)
        if response.ok:
            return True
        if response.status_code in {403, 405}:
            response = requests.get(url, stream=True, timeout=timeout)
            response.close()
            return response.ok
        return False
    except requests.RequestException:
        return False


def get_available_data_urls(refresh: bool = False) -> Dict[int, str]:
    global _AVAILABLE_DATA_URLS_CACHE

    if _AVAILABLE_DATA_URLS_CACHE is not None and not refresh:
        return dict(_AVAILABLE_DATA_URLS_CACHE)

    detected_urls = dict(BASE_DATA_URLS)
    latest_known_year = max(detected_urls)
    max_probe_year = pd.Timestamp.today().year + 1

    for year in range(latest_known_year + 1, max_probe_year + 1):
        candidate_url = _build_year_url(year)
        if _url_exists(candidate_url):
            detected_urls[year] = candidate_url

    _AVAILABLE_DATA_URLS_CACHE = dict(sorted(detected_urls.items()))
    return dict(_AVAILABLE_DATA_URLS_CACHE)

## 3b. Local Data Cache

To avoid downloading ~500 MB of CSVs from the Ministry of Health S3 bucket on every dashboard load, we cache the processed weekly and raw DataFrames as local **parquet** files inside `.data_cache/`.

- `_save_cache()` — writes DataFrames + metadata (timestamp, years covered)
- `_load_cache()` — reads them back instantly
- `cache_is_fresh()` — checks if cache exists and is less than `max_age_hours` old (default: 12h)

The `build_weekly_series()` function checks the cache first. If it's fresh, it returns the cached data in under a second. If not (or if `refresh=True`), it re-downloads everything and updates the cache.

In [ ]:
import json as _json
import time as _time

def _ensure_cache_dir():
    """Create the cache directory if it doesn't exist."""
    _os.makedirs(CACHE_DIR, exist_ok=True)


def _save_cache(raw_all: pd.DataFrame, full_df: pd.DataFrame, df: pd.DataFrame, uf_code: int = TARGET_UF_CODE) -> None:
    """Persist processed DataFrames to local parquet + metadata JSON."""
    _ensure_cache_dir()
    paths = _uf_cache_paths(uf_code)
    full_df.to_parquet(paths["weekly"], index=False)
    raw_cols = ["DT_SIN_PRI", "SEM_PRI", "SG_UF", "ID_MN_RESI", "CLASSI_FIN", "week_start", "notifications"]
    save_cols = [c for c in raw_cols if c in raw_all.columns]
    raw_all[save_cols].to_parquet(paths["raw"], index=False)
    meta = {
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "created_local": pd.Timestamp.now().isoformat(),
        "latest_date": str(df["date"].max().date()),
        "n_weeks": len(df),
        "n_raw": len(raw_all),
        "years": sorted(full_df["year"].unique().tolist()),
        "uf_code": uf_code,
        "uf_label": UF_OPTIONS.get(uf_code, f"UF {uf_code}"),
    }
    with open(paths["meta"], "w") as f:
        _json.dump(meta, f, indent=2)


def _load_cache(uf_code: int = TARGET_UF_CODE):
    """Load cached DataFrames. Returns (raw_all, full_df, df) or None if cache missing."""
    paths = _uf_cache_paths(uf_code)
    weekly_path = paths["weekly"]
    raw_path = paths["raw"]
    # Legacy fallback: if per-UF file missing but old-format exists for default UF
    if not _os.path.exists(weekly_path) and uf_code == TARGET_UF_CODE and _os.path.exists(WEEKLY_CACHE_PATH):
        weekly_path = WEEKLY_CACHE_PATH
        raw_path = RAW_CACHE_PATH
    if not _os.path.exists(weekly_path):
        return None
    try:
        full_df = pd.read_parquet(weekly_path)
        full_df["date"] = pd.to_datetime(full_df["date"])
        raw_all = pd.read_parquet(raw_path) if _os.path.exists(raw_path) else pd.DataFrame()
        if "DT_SIN_PRI" in raw_all.columns:
            raw_all["DT_SIN_PRI"] = pd.to_datetime(raw_all["DT_SIN_PRI"], errors="coerce")
        df = full_df.loc[full_df["date"] >= ANALYSIS_START].copy().reset_index(drop=True)
        return raw_all, full_df, df
    except Exception:
        return None


def cache_is_fresh(max_age_hours: float = 12.0, uf_code: int = TARGET_UF_CODE) -> bool:
    """Return True if a local cache exists and is younger than max_age_hours."""
    paths = _uf_cache_paths(uf_code)
    meta_path = paths["meta"]
    if not _os.path.exists(meta_path) and uf_code == TARGET_UF_CODE and _os.path.exists(CACHE_META_PATH):
        meta_path = CACHE_META_PATH
    if not _os.path.exists(meta_path):
        return False
    try:
        with open(meta_path) as f:
            meta = _json.load(f)
        created = pd.Timestamp(meta["created_utc"])
        age_hours = (pd.Timestamp.utcnow() - created).total_seconds() / 3600
        return age_hours < max_age_hours
    except Exception:
        return False


def get_cache_info(uf_code: int = TARGET_UF_CODE) -> Dict[str, Any]:
    """Return metadata about the current cache, or empty dict if none."""
    paths = _uf_cache_paths(uf_code)
    meta_path = paths["meta"]
    if not _os.path.exists(meta_path) and uf_code == TARGET_UF_CODE and _os.path.exists(CACHE_META_PATH):
        meta_path = CACHE_META_PATH
    if not _os.path.exists(meta_path):
        return {}
    try:
        with open(meta_path) as f:
            return _json.load(f)
    except Exception:
        return {}

## 4. Data Loading — Single Year

`load_dengue_year()` downloads one year's CSV from the Ministry of Health and:
1. Reads only the columns we need (`DT_SIN_PRI`, `SEM_PRI`, `SG_UF`, `ID_MN_RESI`, `CLASSI_FIN`)
2. Filters to Espírito Santo (UF code 32)
3. Computes `week_start` (Monday of each notification's week)
4. Aggregates into a **weekly notification count**

Returns both the raw individual-notification DataFrame and the weekly aggregated one.

In [ ]:
def load_dengue_year(year: int, uf_code: int = TARGET_UF_CODE):
    data_urls = get_available_data_urls()
    usecols = ["DT_SIN_PRI", "SEM_PRI", "SG_UF", "ID_MN_RESI", "CLASSI_FIN"]
    year_df = pd.read_csv(
        data_urls[year],
        compression="zip",
        encoding="latin1",
        usecols=usecols,
        low_memory=False,
    )
    year_df["DT_SIN_PRI"] = pd.to_datetime(year_df["DT_SIN_PRI"], errors="coerce")
    year_df = year_df.dropna(subset=["DT_SIN_PRI"])
    year_df = year_df[year_df["SG_UF"] == uf_code].copy()
    year_df["week_start"] = year_df["DT_SIN_PRI"] - pd.to_timedelta(
        year_df["DT_SIN_PRI"].dt.weekday, unit="D"
    )
    year_df["notifications"] = 1
    weekly = (
        year_df.groupby("week_start", as_index=False)["notifications"]
        .sum()
        .sort_values("week_start")
    )
    weekly["source_year"] = year
    return year_df, weekly

## 5. Data Loading — Full Weekly Series (with Cache)

`build_weekly_series()` loads **all available years**, concatenates them, and:
1. **First checks the local parquet cache** — if it's fresh (< 12 hours old) and `refresh=False`, returns instantly
2. Otherwise downloads all CSVs, merges overlapping week counts, fills gaps, and **saves to cache**
3. Creates a complete weekly grid (every Monday) so there are no gaps
4. Trims to the analysis window (from `ANALYSIS_START` onward)

Returns three DataFrames: raw notifications, full unfiltered weekly series, and the trimmed analysis series.

In [ ]:
def build_weekly_series(refresh: bool = False, uf_code: int = TARGET_UF_CODE):
    # ── Try local cache first ──
    if not refresh and cache_is_fresh(uf_code=uf_code):
        cached = _load_cache(uf_code=uf_code)
        if cached is not None:
            return cached

    # ── Download from S3 CSVs ──
    data_urls = get_available_data_urls(refresh=refresh)
    raw_parts, weekly_parts = [], []
    for year in sorted(data_urls):
        raw_df, weekly_df = load_dengue_year(year, uf_code)
        raw_parts.append(raw_df)
        weekly_parts.append(weekly_df)

    raw_all = pd.concat(raw_parts, ignore_index=True)
    weekly_all = (
        pd.concat(weekly_parts, ignore_index=True)
        .groupby("week_start", as_index=False)["notifications"]
        .sum()
        .rename(columns={"week_start": "date"})
        .sort_values("date")
        .reset_index(drop=True)
    )
    full_weeks = pd.DataFrame(
        {"date": pd.date_range(weekly_all["date"].min(), weekly_all["date"].max(), freq="W-MON")}
    )
    df = full_weeks.merge(weekly_all, on="date", how="left")
    df["is_imputed_zero_week"] = df["notifications"].isna()
    df["notifications"] = df["notifications"].fillna(0).astype(int)
    df["year"] = df["date"].dt.year
    df["epi_week"] = df["date"].dt.isocalendar().week.astype(int)
    full_df = df.copy()
    df = df.loc[df["date"] >= ANALYSIS_START].copy().reset_index(drop=True)

    # ── Save to cache ──
    _save_cache(raw_all, full_df, df, uf_code=uf_code)

    return raw_all, full_df, df

## 6. Feature Engineering

`create_features()` builds the feature columns that all models use:

| Feature | Description |
|---|---|
| `lag1` | Notifications from 1 week ago (most recent known value) |
| `lag4` | Notifications from 4 weeks ago (captures short-term trend) |
| `lag52` | Notifications from 52 weeks ago (same week last year — seasonal signal) |
| `week_sin` | Sine encoding of the epidemiological week number (cyclical seasonality) |
| `week_cos` | Cosine encoding of the epidemiological week number (cyclical seasonality) |

Rows with missing lag values (first few weeks) are dropped.

In [ ]:
def create_features(data: pd.DataFrame) -> pd.DataFrame:
    features = data.copy()
    features["lag1"] = features["notifications"].shift(1)
    features["lag4"] = features["notifications"].shift(4)
    features["lag52"] = features["notifications"].shift(52)
    week_number = features["date"].dt.isocalendar().week.astype(int)
    features["week_sin"] = np.sin(2 * np.pi * week_number / 52)
    features["week_cos"] = np.cos(2 * np.pi * week_number / 52)
    return features.dropna(subset=["lag1", "lag4"]).reset_index(drop=True)

## 7. Model Training — Negative Binomial (Optional)

`fit_negative_binomial_model()` fits a **Negative Binomial GLM** using statsmodels. This is a count-data model that's theoretically appropriate for notification counts (it handles overdispersion better than Poisson).

It's wrapped in a try/except in the bundle because it can fail to converge on small or noisy training sets.

In [ ]:
def fit_negative_binomial_model(X: np.ndarray, y: np.ndarray):
    import statsmodels.api as sm
    from statsmodels.discrete.discrete_model import NegativeBinomial as DiscreteNegativeBinomial

    X_sm = sm.add_constant(X, has_constant="add")
    model = DiscreteNegativeBinomial(y, X_sm)
    result = model.fit(disp=0, maxiter=200)
    return result

## 8. Model Training — Full Bundle

`fit_model_bundle()` trains **all models** on a given feature DataFrame and returns them as a dictionary (the "bundle"):

| Model | Type | Description |
|---|---|---|
| **Naive** | Baseline | Predicts last week's value (random walk) |
| **Seasonal Naive** | Baseline | Predicts same-week-last-year value |
| **Linear Regression** | Learned | Simple linear model on the 4 features |
| **Random Forest** | Learned | 200-tree ensemble, max depth 4, min 5 samples per leaf |
| **Negative Binomial** | Learned (optional) | Count-data GLM, only included if `include_nb=True` and converges |

The baselines don't need fitting — they just use lag values at prediction time.

In [ ]:
def fit_model_bundle(feature_frame: pd.DataFrame, include_nb: bool = True, verbose: bool = False) -> dict[str, dict[str, Any]]:
    X_local = np.asarray(feature_frame[FEATURE_COLS].to_numpy(dtype=float), dtype=float)
    y_local = np.asarray(feature_frame["notifications"].to_numpy(dtype=float), dtype=int)

    bundle: dict[str, dict[str, Any]] = {
        "Naive": {"kind": "baseline"},
        "Seasonal Naive": {"kind": "baseline"},
    }

    linear = LinearRegression()
    linear.fit(X_local, y_local)
    bundle["Linear Regression"] = {"kind": "sklearn", "model": linear}

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
    )
    rf.fit(X_local, y_local)
    bundle["Random Forest"] = {"kind": "sklearn", "model": rf}

    if include_nb:
        try:
            nb_result = fit_negative_binomial_model(X_local, y_local)
            nb_alpha = float(nb_result.params[-1])
            converged = bool(nb_result.mle_retvals.get("converged", False))
            has_finite_bse = np.isfinite(np.asarray(nb_result.bse)).all()
            if not np.isfinite(nb_alpha) or abs(nb_alpha) > 100:
                raise ValueError(f"unstable alpha estimate ({nb_alpha})")
            if not converged:
                raise ValueError("optimizer did not converge")
            if not has_finite_bse:
                raise ValueError("standard errors are not finite")
            bundle["Negative Binomial"] = {
                "kind": "statsmodels",
                "model": nb_result,
                "alpha": nb_alpha,
            }
        except Exception as error:
            if verbose:
                print(f"Negative binomial baseline omitted: {error}")

    return bundle

## 9. Prediction

`predict_with_bundle()` takes a trained bundle and a feature DataFrame, and returns predictions from **every model** in the bundle.

- Naive → just returns `lag1` (last week's value)
- Seasonal Naive → just returns `lag52` (same week last year)
- Linear Regression & Random Forest → uses `.predict()` on the 4 feature columns, clipped to ≥ 0
- Negative Binomial → uses statsmodels `.predict()` with a constant column added

In [ ]:
def predict_with_bundle(bundle: dict[str, dict[str, Any]], feature_frame: pd.DataFrame) -> dict[str, np.ndarray]:
    X_local = np.asarray(feature_frame[FEATURE_COLS].to_numpy(dtype=float), dtype=float)
    predictions: dict[str, np.ndarray] = {
        "Naive": np.asarray(feature_frame["lag1"].to_numpy(dtype=float), dtype=float),
        "Seasonal Naive": np.asarray(feature_frame["lag52"].to_numpy(dtype=float), dtype=float),
        "Linear Regression": np.clip(bundle["Linear Regression"]["model"].predict(X_local), 0, None),
        "Random Forest": np.clip(bundle["Random Forest"]["model"].predict(X_local), 0, None),
    }
    if "Negative Binomial" in bundle:
        import statsmodels.api as sm

        X_local_sm = sm.add_constant(X_local, has_constant="add")
        predictions["Negative Binomial"] = np.clip(
            bundle["Negative Binomial"]["model"].predict(X_local_sm), 0, None
        )
    return predictions

## 10. Evaluation Metrics — Change-Signal Scoring

`score_predictions()` evaluates how well a model predicts **week-over-week direction changes**, not just the raw value. This is critical for surveillance — we care about catching rising weeks.

| Metric | What it measures |
|---|---|
| **Direction Accuracy** | % of weeks where predicted direction (up/down/flat) matched actual |
| **Rising Recall** | Of all truly rising weeks, what % did the model catch? |
| **Rising Precision** | Of all weeks the model said were rising, what % actually were? |
| **Rising F1** | Harmonic mean of precision and recall |
| **False Alarm Rate** | Of all non-rising weeks, what % did the model incorrectly flag as rising? |

In [ ]:
def score_predictions(actual, pred, previous_week):
    actual_direction = np.sign(actual - previous_week)
    predicted_direction = np.sign(pred - previous_week)
    actual_rising = actual > previous_week
    predicted_rising = pred > previous_week
    true_positives = (actual_rising & predicted_rising).sum()
    rising_recall = true_positives / max(1, actual_rising.sum())
    rising_precision = true_positives / max(1, predicted_rising.sum())
    rising_f1 = 2 * rising_precision * rising_recall / max(1e-9, rising_precision + rising_recall)
    actual_not_rising = ~actual_rising
    false_alarms = (actual_not_rising & predicted_rising).sum()
    false_alarm_rate = false_alarms / max(1, actual_not_rising.sum())
    return {
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "Direction Acc.": (actual_direction == predicted_direction).mean(),
        "Rising Recall": rising_recall,
        "Rising Precision": rising_precision,
        "Rising F1": rising_f1,
        "False Alarm Rate": false_alarm_rate,
    }

## 11. Expanding-Window Backtest

`run_backtest()` is the **core evaluation engine**. It simulates what would have happened if we had been using these models in real time:

1. Start with the first 26 weeks as training data
2. Train all models on weeks 1–26, predict week 27
3. Expand training to weeks 1–27, predict week 28
4. Continue until the end of the data

This is called an **expanding-window** (or anchored walk-forward) backtest. It never uses future data to make predictions, so it's an honest estimate of real-world performance.

Returns a DataFrame with one row per tested week, containing the actual value and each model's prediction.

In [ ]:
def run_backtest(features_df: pd.DataFrame, include_nb: bool = True, min_backtest_train: int = MIN_BACKTEST_TRAIN):
    model_cols = {
        "Naive": "naive_pred",
        "Seasonal Naive": "seasonal_naive_pred",
        "Linear Regression": "linear_pred",
        "Random Forest": "rf_pred",
    }
    backtest_frames = []
    for current_idx in range(min_backtest_train, len(features_df)):
        fold_train = pd.DataFrame(features_df.iloc[:current_idx].copy())
        fold_test = pd.DataFrame(features_df.iloc[[current_idx]].copy())
        fold_bundle = fit_model_bundle(fold_train, include_nb=include_nb, verbose=False)
        fold_predictions = predict_with_bundle(fold_bundle, fold_test)
        fold_frame = pd.DataFrame(
            {
                "date": fold_test["date"].values,
                "actual": fold_test["notifications"].values.astype(int),
                "previous_week": fold_test["lag1"].values,
                "naive_pred": fold_predictions["Naive"],
                "seasonal_naive_pred": fold_predictions["Seasonal Naive"],
                "linear_pred": fold_predictions["Linear Regression"],
                "rf_pred": fold_predictions["Random Forest"],
            }
        )
        if "Negative Binomial" in fold_predictions:
            fold_frame["nb_pred"] = fold_predictions["Negative Binomial"]
        backtest_frames.append(fold_frame)

    backtest_results = pd.concat(backtest_frames, ignore_index=True)
    if "nb_pred" in backtest_results.columns:
        model_cols["Negative Binomial"] = "nb_pred"
    return backtest_results, model_cols

## 12. Next-Week Feature Builder

`build_next_week_features()` creates the feature row for the **single next unobserved week**. This is used when the forecast horizon is 1 (simple one-step-ahead forecast).

It looks up `lag1` (last known week), `lag4` (4 weeks back), `lag52` (same week last year), and computes the cyclical week encoding.

In [ ]:
def build_next_week_features(history: pd.DataFrame) -> pd.DataFrame:
    next_date = history["date"].max() + pd.Timedelta(weeks=1)
    lag52_series = history.loc[
        history["date"] == next_date - pd.Timedelta(weeks=52),
        "notifications",
    ]
    if lag52_series.empty:
        raise ValueError("Need at least 52 weeks of history for seasonal features.")

    next_week_number = int(next_date.isocalendar().week)
    return pd.DataFrame(
        {
            "date": [next_date],
            "lag1": [history["notifications"].iloc[-1]],
            "lag4": [history["notifications"].iloc[-4]],
            "lag52": [lag52_series.iloc[0]],
            "week_sin": [np.sin(2 * np.pi * next_week_number / 52)],
            "week_cos": [np.cos(2 * np.pi * next_week_number / 52)],
        }
    )

## 13. Forecast Stabilization Helpers

These three small functions keep multi-step forecasts from exploding or oscillating:

- **`_stabilize_forecast()`** — blends the raw model prediction with an anchor value (weighted average of recent levels and seasonality), then clips the result so it can't jump more than `max_step_delta` from the previous value or exceed the ceiling.

- **`_bounded_seasonal_anchor()`** — computes a seasonal reference that's capped relative to recent activity (prevents the seasonal signal from pulling the forecast unrealistically high).

- **`_estimate_recent_trend()`** — estimates the per-week trend by comparing the mean of the last 4 weeks to the mean of weeks 5–8.

In [ ]:
def _stabilize_forecast(raw_value: float, last_value: float, anchor_value: float, max_step_delta: float, blend_weight: float, ceiling: float) -> float:
    blended = (1.0 - blend_weight) * raw_value + blend_weight * anchor_value
    lower = max(0.0, last_value - max_step_delta)
    upper = min(ceiling, last_value + max_step_delta)
    return float(np.clip(blended, lower, upper))


def _bounded_seasonal_anchor(lag1: float, lag4: float, lag52: float, typical_step_delta: float) -> float:
    recent_anchor = 0.65 * lag1 + 0.35 * lag4
    seasonal_cap = max(recent_anchor * 2.2, recent_anchor + 3.0 * typical_step_delta)
    return float(min(lag52, seasonal_cap))


def _estimate_recent_trend(series: pd.Series) -> float:
    recent_short = series.tail(4)
    recent_prev = series.tail(8).head(4)
    if len(recent_short) == 4 and len(recent_prev) == 4:
        return float((recent_short.mean() - recent_prev.mean()) / 4.0)
    return 0.0

## 14. Multi-Step Display Model Selector

`choose_multi_step_display_model()` picks which model's forecast to **show on the monitoring card** when forecasting multiple weeks ahead.

It penalizes models whose multi-step trajectories look unrealistic:
- Forecasts that spike far above the latest actual value
- Large week-to-week jumps
- Final values far from the starting point

The penalty multiplies the model's backtest score (MAE or composite selection score), so a model that's accurate in backtesting but produces wild multi-step forecasts gets downranked.

In [ ]:
def choose_multi_step_display_model(
    ranking_frame: pd.DataFrame,
    forecast_df: pd.DataFrame,
    latest_actual: float,
    preferred_model: str,
) -> str:
    baseline_models = {"Naive", "Seasonal Naive"}
    candidate_models = [
        model_name
        for model_name in ranking_frame.get("Model", pd.Series(dtype=object)).tolist()
        if model_name in forecast_df.columns and model_name not in baseline_models
    ]
    if not candidate_models:
        return preferred_model

    latest_scale = max(float(latest_actual), 1.0)
    scored_candidates = []
    for model_name in candidate_models:
        series = np.asarray(forecast_df[model_name].to_numpy(dtype=float), dtype=float)
        if len(series) == 0 or not np.isfinite(series).all():
            continue

        max_ratio = float(series.max()) / latest_scale
        final_ratio = float(series[-1]) / latest_scale
        jump_ratio = float(np.abs(np.diff(np.concatenate([[latest_actual], series]))).max()) / latest_scale
        penalty = (
            1.0
            + max(0.0, final_ratio - 2.5)
            + 0.5 * max(0.0, jump_ratio - 1.0)
            + 0.25 * max(0.0, max_ratio - 3.0)
        )

        if "Selection Score" in ranking_frame.columns:
            base_series = ranking_frame.loc[ranking_frame["Model"] == model_name, "Selection Score"]
        elif "MAE" in ranking_frame.columns:
            base_series = ranking_frame.loc[ranking_frame["Model"] == model_name, "MAE"]
        else:
            base_series = pd.Series(dtype=float)
        base_value = float(base_series.iloc[0]) if len(base_series) > 0 else float("inf")
        scored_candidates.append((base_value * penalty, base_value, model_name))

    if not scored_candidates:
        return preferred_model

    scored_candidates.sort()
    return str(scored_candidates[0][2])

## 15. Multi-Step Forecasting Engine

`build_multi_step_forecast()` is the **heart of the forecasting system**. It generates predictions for multiple weeks into the future using a **recursive (autoregressive) approach**:

1. For each future step, it builds features from the most recent known/predicted values
2. Each model makes a raw prediction
3. For learned models (LR, RF), the raw prediction is **stabilized** by blending with an anchor (weighted mix of recent levels, seasonal signal, and trend)
4. The stabilized value is then fed back as `lag1` for the next step

Key stabilization mechanisms:
- **Anchor blending** — the further out we forecast, the more weight goes to the anchor vs. the raw model output
- **Delta limiting** — no single step can change by more than a data-driven maximum (based on the 90th percentile of recent week-to-week changes)
- **Ceiling** — forecasts can't exceed ~3x the latest value or ~1.4x the recent 26-week peak

The Naive model always predicts `lag1` (flat line), and Seasonal Naive uses a bounded seasonal anchor.

In [ ]:
def build_multi_step_forecast(history: pd.DataFrame, bundle: Dict[str, Dict[str, Any]], horizon: int) -> pd.DataFrame:
    if horizon < 1:
        raise ValueError("horizon must be at least 1")

    model_names = list(bundle.keys())
    base = history[["date", "notifications"]].copy().reset_index(drop=True)
    model_series = {name: pd.DataFrame(base.copy()) for name in model_names}
    rows: List[Dict[str, Union[float, int, pd.Timestamp]]] = []

    recent_diffs = history["notifications"].diff().dropna().tail(min(26, max(1, len(history) - 1)))
    typical_step_delta = float(np.quantile(np.abs(recent_diffs), 0.9)) if len(recent_diffs) > 0 else 25.0
    typical_step_delta = max(typical_step_delta, 20.0)
    seasonal_profile = (
        history.groupby(history["date"].dt.isocalendar().week.astype(int))["notifications"]
        .median()
        .to_dict()
    )
    hist_max = float(history["notifications"].max())
    recent_peak = float(history["notifications"].tail(26).max()) if len(history) > 0 else hist_max
    ceiling = max(hist_max * 0.9, recent_peak * 1.4, float(history["notifications"].iloc[-1]) * 3.0, 75.0)

    for step in range(1, horizon + 1):
        next_date = history["date"].max() + pd.Timedelta(weeks=step)
        week_number = int(next_date.isocalendar().week)
        row: Dict[str, Union[float, int, pd.Timestamp]] = {"date": next_date, "step": step}
        blend_weight = min(0.82, 0.30 + 0.10 * max(0, step - 1))

        for model_name in model_names:
            series = model_series[model_name]
            lag1 = float(series["notifications"].iloc[-1])
            lag4 = float(series["notifications"].iloc[-4]) if len(series) >= 4 else lag1
            recent_level = float(series["notifications"].tail(8).median())
            trend_per_week = _estimate_recent_trend(series["notifications"])
            lag52_date = next_date - pd.Timedelta(weeks=52)
            lag52_series = series.loc[series["date"] == lag52_date, "notifications"]
            lag52 = float(lag52_series.iloc[0]) if len(lag52_series) > 0 else lag1
            seasonal_median = float(seasonal_profile.get(week_number, lag52))

            feat = pd.DataFrame(
                {
                    "date": [next_date],
                    "lag1": [lag1],
                    "lag4": [lag4],
                    "lag52": [lag52],
                    "week_sin": [np.sin(2 * np.pi * week_number / 52)],
                    "week_cos": [np.cos(2 * np.pi * week_number / 52)],
                }
            )
            raw_predictions = predict_with_bundle(bundle, feat)
            raw_value = max(0.0, float(raw_predictions[model_name][0]))
            recent_anchor = 0.55 * lag1 + 0.20 * lag4 + 0.25 * recent_level
            seasonal_reference = 0.5 * lag52 + 0.5 * seasonal_median
            seasonal_cap = max(recent_anchor * 1.45, recent_anchor + 1.75 * typical_step_delta)
            seasonal_anchor = float(min(seasonal_reference, seasonal_cap))
            trend_target = max(0.0, recent_anchor + trend_per_week * min(step, 4))
            anchor_value = 0.65 * recent_anchor + 0.20 * seasonal_anchor + 0.15 * trend_target
            delta_limit = max(typical_step_delta * (0.65 + 0.20 * math.sqrt(step)), 25.0)

            if model_name == "Naive":
                forecast_value = lag1
            elif model_name == "Seasonal Naive":
                forecast_value = seasonal_anchor
            else:
                forecast_value = _stabilize_forecast(
                    raw_value=raw_value,
                    last_value=lag1,
                    anchor_value=anchor_value,
                    max_step_delta=delta_limit,
                    blend_weight=min(0.92, 0.55 + 0.08 * max(0, step - 1)),
                    ceiling=ceiling,
                )

            row[model_name] = forecast_value
            model_series[model_name] = pd.concat(
                [series, pd.DataFrame({"date": [next_date], "notifications": [forecast_value]})],
                ignore_index=True,
            )
        rows.append(row)

    return pd.DataFrame(rows)

## 16. Error Tables & Holdout Management

Utility functions for model evaluation:

- **`build_error_table()`** — computes MAE and RMSE for each model on a given backtest slice, returns a sorted DataFrame
- **`determine_holdout_start()`** — finds the date that splits the backtest into training and holdout periods (default: last 52 weeks are holdout, minimum 26 weeks)

In [ ]:
def build_error_table(bt_slice: pd.DataFrame, model_cols: Dict[str, str]) -> pd.DataFrame:
    rows: List[Dict[str, Union[str, float]]] = []
    for model_name, col in model_cols.items():
        sample = bt_slice.dropna(subset=[col])
        if len(sample) == 0:
            continue
        rows.append(
            {
                "Model": model_name,
                "MAE": float(mean_absolute_error(sample["actual"], sample[col])),
                "RMSE": float(np.sqrt(mean_squared_error(sample["actual"], sample[col]))),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["Model", "MAE", "RMSE"])
    return pd.DataFrame(rows).sort_values(by=["MAE", "RMSE"]).reset_index(drop=True)


def determine_holdout_start(
    backtest_results: pd.DataFrame,
    holdout_weeks: int = DEFAULT_HOLDOUT_WEEKS,
    minimum_holdout_weeks: int = 26,
) -> pd.Timestamp:
    unique_dates = sorted(pd.unique(backtest_results["date"]))
    if len(unique_dates) == 0:
        raise ValueError("Backtest results are empty; cannot determine holdout start.")

    effective_holdout_weeks = min(len(unique_dates), max(minimum_holdout_weeks, holdout_weeks))
    return pd.Timestamp(unique_dates[-effective_holdout_weeks])

## 17. Predictor Diagnostics

`build_predictor_diagnostics()` creates a summary table for the production model showing:
- Which model is in production
- The holdout window start date
- The latest tested week's actual vs. predicted values
- Recent 8-week holdout MAE

This is used in the dashboard's model notes section.

In [ ]:
def build_predictor_diagnostics(
    backtest_results: pd.DataFrame,
    model_cols: Dict[str, str],
    prod_name: str,
    holdout_start: pd.Timestamp,
) -> pd.DataFrame:
    prod_col = model_cols[prod_name]
    prod_backtest = backtest_results.dropna(subset=[prod_col]).copy().reset_index(drop=True)
    latest_eval = prod_backtest.iloc[-1]
    latest_abs_error = abs(float(latest_eval["actual"]) - float(latest_eval[prod_col]))
    recent_holdout = prod_backtest[prod_backtest["date"] >= holdout_start].tail(8)
    recent_holdout_mae = (
        float(mean_absolute_error(recent_holdout["actual"], recent_holdout[prod_col]))
        if len(recent_holdout) > 0
        else float("nan")
    )

    return pd.DataFrame(
        {
            "Metric": [
                "Production model",
                "Testing window start",
                "Latest tested week",
                "Latest tested actual",
                "Latest tested prediction",
                "Latest tested absolute error",
                "Recent holdout MAE",
            ],
            "Value": [
                prod_name,
                str(pd.Timestamp(holdout_start).date()),
                str(pd.Timestamp(latest_eval["date"]).date()),
                f"{int(latest_eval['actual']):,}",
                f"{float(latest_eval[prod_col]):.1f}",
                f"{latest_abs_error:.1f}",
                f"{recent_holdout_mae:.1f}" if np.isfinite(recent_holdout_mae) else "N/A",
            ],
        }
    )

## 18. Composite Model Selection Scoring

`build_selection_score_table()` computes a **composite score** to rank models for production use. It weighs multiple criteria:

| Weight | Component | Rationale |
|---|---|---|
| 45% | Recent MAE | Raw accuracy is most important |
| 20% | Latest absolute error | How well the model did on the very last week |
| 20% | 1 − Direction accuracy | Penalize models that miss the trend direction |
| 20% | 1 − Rising F1 | Penalize models that miss rising weeks |
| 15% | False alarm rate | Penalize models that cry wolf |

Lower composite score = better model. The best-scoring model becomes the production model.

In [ ]:
def build_selection_score_table(
    evaluation_bt: pd.DataFrame,
    model_cols: Dict[str, str],
) -> pd.DataFrame:
    rows: List[Dict[str, Union[str, float]]] = []
    for model_name, col in model_cols.items():
        sample = evaluation_bt.dropna(subset=[col]).copy()
        if len(sample) == 0:
            continue

        score = score_predictions(
            sample["actual"].values,
            sample[col].values,
            sample["previous_week"].values,
        )
        recent_mae = float(mean_absolute_error(sample["actual"], sample[col]))
        latest_abs_error = abs(float(sample.iloc[-1]["actual"]) - float(sample.iloc[-1][col]))
        rows.append(
            {
                "Model": model_name,
                "Recent MAE": recent_mae,
                "Latest Abs Error": latest_abs_error,
                "Direction Acc.": float(score["Direction Acc."]),
                "Rising F1": float(score["Rising F1"]),
                "False Alarm Rate": float(score["False Alarm Rate"]),
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "Model",
                "Recent MAE",
                "Latest Abs Error",
                "Direction Acc.",
                "Rising F1",
                "False Alarm Rate",
                "Selection Score",
            ]
        )

    score_table = pd.DataFrame(rows)
    mae_scale = max(float(score_table["Recent MAE"].median()), 1.0)
    latest_scale = max(float(score_table["Latest Abs Error"].median()), 1.0)
    score_table["Selection Score"] = (
        0.45 * (score_table["Recent MAE"] / mae_scale)
        + 0.20 * (score_table["Latest Abs Error"] / latest_scale)
        + 0.20 * (1.0 - score_table["Direction Acc."])
        + 0.20 * (1.0 - score_table["Rising F1"])
        + 0.15 * score_table["False Alarm Rate"]
    )
    score_order = list(np.argsort(score_table["Selection Score"].to_numpy(dtype=float)))
    return pd.DataFrame(score_table.iloc[score_order].copy()).reset_index(drop=True)

## 19. Master Orchestrator — `build_monitoring_state()`

This is the **main entry point** for the dashboard. It chains every step above into one call:

1. Loads (or fetches) the data
2. Engineers features
3. Runs the full expanding-window backtest
4. Splits into **selection** (pre-holdout) and **holdout** periods
5. Scores models on the pre-holdout selection window (no holdout leakage)
6. Picks a **production model** (best selection score) and a **display model** (for multi-step chart)
7. Generates a multi-step forecast
8. Derives trend, risk, safety bands, and diagnostics

**Key design choices:**
- **Selection vs holdout**: the selection window is the last `selection_weeks` weeks *before* the holdout, never inside it. This prevents information leakage.
- **Production model vs display model**: the production model drives trend/risk labels and safety bands. The display model is chosen for visual stability in multi-step charts (avoids explosive extrapolation). They may differ.
- **NB status tracking**: returns whether Negative Binomial was requested, available, or omitted due to convergence failure.

In [ ]:
def build_monitoring_state(
    horizon: int = 1,
    include_nb: bool = True,
    refresh_data: bool = False,
    holdout_weeks: int = DEFAULT_HOLDOUT_WEEKS,
    selection_weeks: int = DEFAULT_SELECTION_WEEKS,
    uf_code: int = TARGET_UF_CODE,
) -> Dict[str, Any]:
    # ── Data loading (uses cache unless refresh_data is True) ──
    available_data_urls = get_available_data_urls(refresh=refresh_data)
    available_years = sorted(available_data_urls)
    raw_all, full_df, df = build_weekly_series(refresh=refresh_data, uf_code=uf_code)
    features_df = create_features(df)
    backtest_results, model_cols = run_backtest(features_df, include_nb=include_nb)

    # ── Track NB model status ──
    nb_requested = include_nb
    nb_available = "Negative Binomial" in model_cols
    if nb_requested and not nb_available:
        nb_status = "omitted"  # requested but fit did not converge
    elif nb_requested and nb_available:
        nb_status = "active"
    else:
        nb_status = "off"

    # ── Holdout / selection split ──
    full_err = build_error_table(backtest_results, model_cols)
    holdout_start = determine_holdout_start(backtest_results, holdout_weeks=holdout_weeks)
    selection_bt = backtest_results[backtest_results["date"] < holdout_start].copy()
    holdout_bt = backtest_results[backtest_results["date"] >= holdout_start].copy()
    holdout_err = build_error_table(holdout_bt, model_cols) if len(holdout_bt) > 0 else full_err.copy()

    # ── Selection period: last N weeks BEFORE the holdout (no leakage) ──
    recent_selection_bt = selection_bt.tail(selection_weeks).copy() if len(selection_bt) > 0 else selection_bt.copy()
    recent_err = build_error_table(recent_selection_bt, model_cols) if len(recent_selection_bt) > 0 else holdout_err.copy()
    selection_score_table = build_selection_score_table(recent_selection_bt, model_cols) if len(recent_selection_bt) > 0 else pd.DataFrame()

    # ── Model ranking ──
    ranking_frame = pd.DataFrame()
    if len(selection_score_table) > 0:
        ranking_frame = selection_score_table[["Model", "Selection Score"]].copy()
        ranking_basis = f"Pre-holdout selection score ({len(recent_selection_bt)} weeks before holdout)"
    elif len(recent_err) > 0:
        ranking_frame = recent_err
        ranking_basis = f"Pre-holdout selection focus ({len(recent_selection_bt)} weeks)"
    elif len(holdout_err) > 0:
        ranking_frame = holdout_err
        ranking_basis = f"Holdout evaluation ({len(holdout_bt)} weeks)"
    else:
        ranking_frame = full_err
        ranking_basis = "Full expanding-window backtest"

    ranking_frame = pd.DataFrame(ranking_frame).reset_index(drop=True)
    prod_name = ranking_frame.iloc[0]["Model"] if len(ranking_frame) > 0 else "Naive"
    prod_col = model_cols[prod_name]
    learned_models = [m for m in model_cols if m not in {"Naive", "Seasonal Naive"}]
    learned_ranking_source = recent_err if len(recent_err) > 0 else holdout_err if len(holdout_err) > 0 else full_err
    prod_learned_ranking = learned_ranking_source[learned_ranking_source["Model"].isin(learned_models)].copy()
    if len(prod_learned_ranking) > 0:
        learned_order = np.argsort(prod_learned_ranking["MAE"].to_numpy(dtype=float))
        prod_learned_ranking = prod_learned_ranking.iloc[learned_order].reset_index(drop=True)
    if prod_name in learned_models and len(prod_learned_ranking) > 1:
        prod_learned_name = str(prod_learned_ranking.iloc[1]["Model"])
    elif prod_name not in learned_models and len(prod_learned_ranking) > 0:
        prod_learned_name = str(prod_learned_ranking.iloc[0]["Model"])
    else:
        prod_learned_name = prod_name

    # ── Forecasting ──
    prod_bundle = fit_model_bundle(features_df, include_nb=include_nb, verbose=False)
    if horizon <= 1:
        next_features = build_next_week_features(df)
        next_predictions = predict_with_bundle(prod_bundle, next_features)
        next_date = pd.Timestamp(next_features["date"].iloc[0])
        forecast_df = pd.DataFrame(
            {
                "date": [next_date],
                "step": [1],
                **{name: [float(values[0])] for name, values in next_predictions.items()},
            }
        )
    else:
        forecast_df = build_multi_step_forecast(df, prod_bundle, horizon)
        next_date = pd.Timestamp(forecast_df["date"].iloc[0])

    # ── Display model: used for multi-step chart visualisation ──
    if horizon > 1:
        display_model = choose_multi_step_display_model(
            ranking_frame,
            forecast_df,
            latest_actual=float(df["notifications"].iloc[-1]),
            preferred_model=prod_learned_name if prod_name in {"Naive", "Seasonal Naive"} else prod_name,
        )
    else:
        display_model = prod_name

    # ── Safety bands (always based on production model residuals) ──
    residual_source = holdout_bt if len(holdout_bt) >= 8 else backtest_results
    residuals = residual_source["actual"].values - residual_source[prod_col].values
    valid_residuals = residuals[np.isfinite(residuals)]
    r_lo, r_hi = np.quantile(valid_residuals, BAND_QUANTILES)
    steps = forecast_df["step"].values
    forecast_df["lower"] = np.clip(forecast_df[display_model].values + r_lo * np.sqrt(steps), 0, None)
    forecast_df["upper"] = np.clip(forecast_df[display_model].values + r_hi * np.sqrt(steps), 0, None)

    # ── Headline values ──
    latest_actual = int(df["notifications"].iloc[-1])
    latest_date = pd.Timestamp(df["date"].iloc[-1])
    # prod_forecast: the production model's next-step forecast (for trend/risk)
    prod_forecast = float(forecast_df[prod_name].iloc[0]) if prod_name in forecast_df.columns else float(forecast_df[display_model].iloc[0])
    # display_forecast: the display model's next-step forecast (for chart headline)
    display_forecast = float(forecast_df[display_model].iloc[0])
    learned_forecast = float(forecast_df[prod_learned_name].iloc[0])
    next_lower = float(forecast_df["lower"].iloc[0])
    next_upper = float(forecast_df["upper"].iloc[0])
    final_forecast = float(forecast_df[display_model].iloc[-1])

    # ── Trend & Risk: always driven by the PRODUCTION model ──
    pct_change = (prod_forecast - latest_actual) / max(latest_actual, 1)
    if pct_change > TREND_THRESHOLD:
        trend_label, trend_icon = "Rising", "\U0001f4c8"
    elif pct_change < -TREND_THRESHOLD:
        trend_label, trend_icon = "Falling", "\U0001f4c9"
    else:
        trend_label, trend_icon = "Stable", "\u27a1\ufe0f"

    q50, q75, q90 = df["notifications"].quantile([0.50, 0.75, 0.90]).values
    if prod_forecast >= q90:
        risk_label, risk_icon = "Very High", "\U0001f534"
    elif prod_forecast >= q75:
        risk_label, risk_icon = "High", "\U0001f7e0"
    elif prod_forecast >= q50:
        risk_label, risk_icon = "Moderate", "\U0001f7e1"
    else:
        risk_label, risk_icon = "Low", "\U0001f7e2"

    fr_card = holdout_bt.copy()
    fr_card["pl"] = np.clip(fr_card[prod_col] + r_lo, 0, None)
    fr_card["pu"] = np.clip(fr_card[prod_col] + r_hi, 0, None)
    cov = ((fr_card["actual"] >= fr_card["pl"]) & (fr_card["actual"] <= fr_card["pu"])).mean() if len(fr_card) > 0 else np.nan
    recent_8 = holdout_bt.tail(8)
    recent_mae = mean_absolute_error(recent_8["actual"], recent_8[prod_col]) if len(recent_8) > 0 else np.nan
    predictor_diagnostics = build_predictor_diagnostics(backtest_results, model_cols, prod_name, holdout_start)

    # ── Cache metadata for dashboard attribution ──
    _cache_info = get_cache_info(uf_code=uf_code)

    uf_label = UF_OPTIONS.get(uf_code, f"UF {uf_code}")

    return {
        "available_data_urls": available_data_urls,
        "available_years": available_years,
        "raw_all": raw_all,
        "full_df": full_df,
        "df": df,
        "features_df": features_df,
        "backtest_results": backtest_results,
        "model_cols": model_cols,
        "full_err": full_err,
        "holdout_start": holdout_start,
        "selection_bt": selection_bt,
        "holdout_bt": holdout_bt,
        "holdout_err": holdout_err,
        "recent_selection_bt": recent_selection_bt,
        "recent_err": recent_err,
        "selection_score_table": selection_score_table,
        "ranking_frame": ranking_frame,
        "ranking_basis": ranking_basis,
        "prod_name": prod_name,
        "prod_col": prod_col,
        "prod_learned_name": prod_learned_name,
        "prod_learned_ranking": prod_learned_ranking,
        "prod_bundle": prod_bundle,
        "forecast_df": forecast_df,
        "next_date": next_date,
        "display_model": display_model,
        "r_lo": float(r_lo),
        "r_hi": float(r_hi),
        "latest_actual": latest_actual,
        "latest_date": latest_date,
        "prod_forecast": prod_forecast,
        "display_forecast": display_forecast,
        "learned_forecast": learned_forecast,
        "next_lower": next_lower,
        "next_upper": next_upper,
        "final_forecast": final_forecast,
        "pct_change": float(pct_change),
        "trend_label": trend_label,
        "trend_icon": trend_icon,
        "risk_label": risk_label,
        "risk_icon": risk_icon,
        "cov": float(cov) if np.isfinite(cov) else np.nan,
        "recent_mae": float(recent_mae) if np.isfinite(recent_mae) else np.nan,
        "predictor_diagnostics": predictor_diagnostics,
        "nb_status": nb_status,
        "uf_code": uf_code,
        "uf_label": uf_label,
        "data_source": DATA_SOURCE_LABEL,
        "cache_info": _cache_info,
    }

---

## Pipeline Summary

```
SINAN CSVs (S3)  →  load_dengue_year()  →  build_weekly_series()
                                                    ↓
                                           create_features()
                                                    ↓
                                    ┌───── run_backtest() ─────┐
                                    │  (expanding window)      │
                                    │  fit → predict → record  │
                                    └──────────────────────────┘
                                                    ↓
                              build_error_table() + build_selection_score_table()
                                                    ↓
                                      Select production model
                                                    ↓
                              fit_model_bundle() on full training data
                                                    ↓
                          build_multi_step_forecast() or single-step
                                                    ↓
                              Safety bands from residual quantiles
                                                    ↓
                             Trend + Risk labels → Dashboard state
```